<a href="https://colab.research.google.com/github/Tomca765/Tools-for-the-Analysis-of-Histological-Images-of-Liver-Scaffolds-in-the-QuPath-Environment/blob/main/qupath_server.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q flask opencv-python-headless pillow pyngrok

import os
import threading
import cv2
import numpy as np
from flask import Flask, request, jsonify
from pyngrok import ngrok

# =========================================================================
# CONFIGURATION
# =========================================================================
NGROK_AUTH_TOKEN = "3KdNGUA1jz7IBxOfwhLcWXT7moa_36tNzYBAc1rpuoqap4dAX"
NGROK_DOMAIN     = "upstage-justice-oblong.ngrok-free.dev"

TISSUE_MAX_BRIGHTNESS = 245
MIN_TISSUE_SATURATION = 20

MIN_CELL_AREA     = 20000
MAX_CELL_AREA     = 15000000000000000000
MIN_CIRCULARITY   = 0.0001
BLUR_KERNEL_SIZE  = 9
POLYGON_EPSILON   = 0.0006

USE_ADAPTIVE_THRESH = True
ADAPTIVE_BLOCK_SIZE = 31
ADAPTIVE_C          = 3

app = Flask(__name__)

@app.route('/analyze', methods=['POST'])
def analyze_image():
    img_bytes = request.data
    if not img_bytes:
        return jsonify({'status': 'error', 'message': 'No image data'}), 400

    nparr = np.frombuffer(img_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    if img is None:
        return jsonify({'status': 'error', 'message': 'Decode failed'}), 400

    # 1. TISSUE & PINK COLOR MASKS
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    saturation = hsv[:, :, 1]

    tissue_mask = ((gray < TISSUE_MAX_BRIGHTNESS) | (saturation > MIN_TISSUE_SATURATION)).astype(np.uint8) * 255
    kernel_tissue = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    tissue_mask = cv2.morphologyEx(tissue_mask, cv2.MORPH_CLOSE, kernel_tissue)

    # Detekce růžové (Eosin): Hue 135-175 a 0-10
    pink1 = cv2.inRange(hsv, np.array([135, 30, 50], dtype=np.uint8), np.array([175, 255, 255], dtype=np.uint8))
    pink2 = cv2.inRange(hsv, np.array([0, 30, 50], dtype=np.uint8), np.array([10, 255, 255], dtype=np.uint8))
    pink_mask = cv2.bitwise_or(pink1, pink2)

    # 2. CELL SEGMENTATION
    blurred = cv2.GaussianBlur(gray, (BLUR_KERNEL_SIZE, BLUR_KERNEL_SIZE), 0)
    if USE_ADAPTIVE_THRESH:
        binary = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, ADAPTIVE_BLOCK_SIZE, ADAPTIVE_C)
    else:
        _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    cells_only = cv2.bitwise_and(binary, binary, mask=tissue_mask)
    kernel_cells = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    cleaned_cells = cv2.morphologyEx(cells_only, cv2.MORPH_OPEN, kernel_cells)

    contours, _ = cv2.findContours(cleaned_cells, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    polygons = []
    pink_percentages = []

    for cnt in contours:
        area = cv2.contourArea(cnt)
        perimeter = cv2.arcLength(cnt, True)
        if perimeter == 0:
            continue

        circularity = (4 * np.pi * area) / (perimeter ** 2)

        if MIN_CELL_AREA <= area <= MAX_CELL_AREA and circularity >= MIN_CIRCULARITY:
            approx = cv2.approxPolyDP(cnt, POLYGON_EPSILON * perimeter, True)
            pts = [[int(pt[0][0]), int(pt[0][1])] for pt in approx]

            if len(pts) >= 3:
                # Spočítání růžových pixelů uvnitř polygonu buňky
                x, y, w, h = cv2.boundingRect(cnt)
                cell_mask_roi = np.zeros((h, w), dtype=np.uint8)
                cnt_offset = (cnt - [x, y]).astype(np.int32)
                cv2.drawContours(cell_mask_roi, [cnt_offset], -1, 255, thickness=cv2.FILLED)

                total_pixels = cv2.countNonZero(cell_mask_roi)
                if total_pixels > 0:
                    roi_pink = pink_mask[y:y+h, x:x+w]
                    pink_pixels = cv2.countNonZero(cv2.bitwise_and(roi_pink, roi_pink, mask=cell_mask_roi))
                    pink_pct = round((float(pink_pixels) / float(total_pixels)) * 100.0, 2)
                else:
                    pink_pct = 0.0

                polygons.append(pts)
                pink_percentages.append(pink_pct)

    return jsonify({
        'status': 'success',
        'contour_count': len(polygons),
        'contours': polygons,
        'pink_percentages': pink_percentages
    })

# SERVER LAUNCH
ngrok.kill()
ngrok_config = os.path.expanduser("~/.config/ngrok/ngrok.yml")
if os.path.exists(ngrok_config):
    os.remove(ngrok_config)

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
threading.Thread(target=lambda: app.run(host='0.0.0.0', port=5000), daemon=True).start()

tunnel = ngrok.connect(5000, url=NGROK_DOMAIN)
print(f"PERMANENT ENDPOINT ACTIVE: {tunnel.public_url}/analyze")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


PERMANENT ENDPOINT ACTIVE: https://upstage-justice-oblong.ngrok-free.dev/analyze


In [1]:
!pip install -q flask opencv-python-headless pillow pyngrok

import os
import base64
import threading
import cv2
import numpy as np
from flask import Flask, request, jsonify
from pyngrok import ngrok

# =========================================================================
# CONFIGURATION
# =========================================================================
NGROK_AUTH_TOKEN = "3KdNGUA1jz7IBxOfwhLcWXT7moa_36tNzYBAc1rpuoqap4dAX"
NGROK_DOMAIN     = "upstage-justice-oblong.ngrok-free.dev"

# Zastavení o bílé pozadí mimo tkáň
TISSUE_MAX_BRIGHTNESS = 245
MIN_TISSUE_SATURATION = 20

# Parametry nafukování bubliny a zastavení o změnu barvy (CIE Lab)
MAX_COLOR_DIFF    = 22.0   # Max. odchylka barvy oproti tkáni kolem žíly (nižší = zastaví se o jemnější změnu barvy)
MAX_EDGE_STRENGTH = 18.0   # Citlivost na ostré barevné přechody / septa (nižší = přísnější stěna)
SMOOTH_BLUR_SIZE  = 9      # Vyhlazení jemných struktur uvnitř lalůčku

app = Flask(__name__)

@app.route('/analyze', methods=['POST'])
def analyze_image():
    data = request.get_json(silent=True)
    if not data or 'image' not in data or 'seeds' not in data:
        return jsonify({'status': 'error', 'message': 'Missing image or seeds'}), 400

    img_bytes = base64.b64decode(data['image'])
    nparr = np.frombuffer(img_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    if img is None:
        return jsonify({'status': 'error', 'message': 'Decode failed'}), 400

    h, w = img.shape[:2]
    seeds = data['seeds']
    seed_names = data.get('seed_names', [])
    target_index = data.get('target_index', None)  # None = Fáze 1 (vše), int = Fáze 2 (detail jedné žíly)
    num_seeds = len(seeds)

    if num_seeds == 0:
        return jsonify({
            'status': 'success',
            'contour_count': 0,
            'contours': [],
            'pink_percentages': [],
            'seed_names': []
        })

    is_fine_pass = target_index is not None
    poly_epsilon = 0.0004 if is_fine_pass else 0.0012
    ring_width = 16 if is_fine_pass else 8

    # 1. VYHLAZENÍ A PŘEVOD DO CIE LAB
    blur_k = SMOOTH_BLUR_SIZE if SMOOTH_BLUR_SIZE % 2 == 1 else SMOOTH_BLUR_SIZE + 1
    blurred_bgr = cv2.GaussianBlur(img, (blur_k, blur_k), 0)
    lab = cv2.cvtColor(blurred_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)

    gray = cv2.cvtColor(blurred_bgr, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(blurred_bgr, cv2.COLOR_BGR2HSV)
    saturation = hsv[:, :, 1]

    # Maska tkáně (zastaví bubliny o bílé pozadí kolem řezu)
    tissue_mask = ((gray < TISSUE_MAX_BRIGHTNESS) | (saturation > MIN_TISSUE_SATURATION)).astype(np.uint8) * 255
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    tissue_mask = cv2.morphologyEx(tissue_mask, cv2.MORPH_CLOSE, kernel_close)

    # Výpočet barevného gradientu (ostré změny barvy = bariéra)
    grad_sq = np.zeros((h, w), dtype=np.float32)
    for c in range(3):
        gx = cv2.Sobel(lab[:, :, c], cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(lab[:, :, c], cv2.CV_32F, 0, 1, ksize=3)
        grad_sq += (gx * gx + gy * gy) / 16.0
    edge_mag = np.sqrt(grad_sq)
    edge_ok = edge_mag <= MAX_EDGE_STRENGTH

    # 2. PŘÍPRAVA KAŽDÉ CENTRÁLNÍ ŽÍLY (SEED) A JEJÍ BAREVNÉ ZÓNY
    seed_masks = []
    allowed_masks = []
    union_allowed = np.zeros((h, w), dtype=bool)
    ring_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ring_width * 2 + 1, ring_width * 2 + 1))

    for pts in seeds:
        s_mask = np.zeros((h, w), dtype=np.uint8)
        poly = np.array(pts, dtype=np.int32).reshape((-1, 1, 2))
        cv2.fillPoly(s_mask, [poly], 255)
        seed_bool = s_mask > 0
        seed_masks.append(seed_bool)

        if not np.any(seed_bool):
            allowed_masks.append(np.zeros((h, w), dtype=bool))
            continue

        # Odečet referenční barvy tkáně těsně kolem zakroužkované žíly
        dilated = cv2.dilate(s_mask, ring_kernel)
        ring_bool = (dilated > 0) & (~seed_bool) & (tissue_mask > 0)

        # Fallback: pokud byla žíla zakroužkovaná hluboko uvnitř velkého bílého lumen, najdeme nejbližší tkáň
        if not np.any(ring_bool) and np.any(tissue_mask > 0):
            dist_from_seed = cv2.distanceTransform((~seed_bool).astype(np.uint8), cv2.DIST_L2, 5)
            tissue_dists = dist_from_seed[tissue_mask > 0]
            min_d = np.min(tissue_dists)
            ring_bool = (tissue_mask > 0) & (dist_from_seed <= min_d + ring_width)

        if np.any(ring_bool):
            ref_lab = np.median(lab[ring_bool], axis=0)
        else:
            ref_lab = np.median(lab[seed_bool], axis=0)

        # Barevná vzdálenost každého pixelu od tkáně u této žíly
        diff = lab - ref_lab
        color_dist = np.sqrt(np.sum(diff * diff, axis=2))

        raw_allowed = ((tissue_mask > 0) & (color_dist <= MAX_COLOR_DIFF) & edge_ok) | seed_bool

        # Ponechat jen souvislou oblast spojenou s touto žilou
        _, cc_labels = cv2.connectedComponents(raw_allowed.astype(np.uint8), connectivity=4)
        touching = np.unique(cc_labels[seed_bool])
        touching = touching[touching != 0]

        if len(touching) > 0:
            reachable = np.isin(cc_labels, touching)
        else:
            reachable = seed_bool

        allowed_masks.append(reachable)
        union_allowed |= reachable

    # 3. SOUČASNÉ NAFUKOVÁNÍ BUBLIN (MARKER-CONTROLLED WATERSHED)
    barrier_id = num_seeds + 1
    markers = np.full((h, w), barrier_id, dtype=np.int32)
    markers[union_allowed] = 0

    for idx, seed_bool in enumerate(seed_masks):
        markers[seed_bool] = idx + 1

    grad_norm = np.clip((edge_mag / (MAX_EDGE_STRENGTH + 1e-5)) * 255.0, 0, 255).astype(np.uint8)
    grad_bgr = cv2.merge([grad_norm, grad_norm, grad_norm])
    cv2.watershed(grad_bgr, markers)

    # 4. VYTVOŘENÍ OBDRYSŮ A VÝPOČET PINK %
    hsv_orig = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    pink1 = cv2.inRange(hsv_orig, np.array([135, 30, 50], dtype=np.uint8), np.array([175, 255, 255], dtype=np.uint8))
    pink2 = cv2.inRange(hsv_orig, np.array([0, 30, 50], dtype=np.uint8), np.array([10, 255, 255], dtype=np.uint8))
    pink_mask = cv2.bitwise_or(pink1, pink2)

    polygons = []
    pink_percentages = []
    out_names = []
    clean_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

    indices_to_extract = [int(target_index)] if is_fine_pass else list(range(num_seeds))

    for idx in indices_to_extract:
        if idx < 0 or idx >= num_seeds:
            continue

        vein_id = idx + 1
        region = ((markers == vein_id) & allowed_masks[idx]) | seed_masks[idx]
        region_u8 = cv2.morphologyEx(region.astype(np.uint8) * 255, cv2.MORPH_CLOSE, clean_kernel)

        contours, _ = cv2.findContours(region_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue

        cnt = max(contours, key=cv2.contourArea)
        perimeter = cv2.arcLength(cnt, True)
        if perimeter == 0:
            continue

        approx = cv2.approxPolyDP(cnt, poly_epsilon * perimeter, True)
        pts = [[int(pt[0][0]), int(pt[0][1])] for pt in approx]

        if len(pts) >= 3:
            x, y, bw, bh = cv2.boundingRect(cnt)
            cell_mask_roi = np.zeros((bh, bw), dtype=np.uint8)
            cnt_offset = (cnt - [x, y]).astype(np.int32)
            cv2.drawContours(cell_mask_roi, [cnt_offset], -1, 255, thickness=cv2.FILLED)

            total_pixels = cv2.countNonZero(cell_mask_roi)
            if total_pixels > 0:
                roi_pink = pink_mask[y:y+bh, x:x+bw]
                pink_pixels = cv2.countNonZero(cv2.bitwise_and(roi_pink, roi_pink, mask=cell_mask_roi))
                pink_pct = round((float(pink_pixels) / float(total_pixels)) * 100.0, 2)
            else:
                pink_pct = 0.0

            polygons.append(pts)
            pink_percentages.append(pink_pct)
            name = seed_names[idx] if idx < len(seed_names) and seed_names[idx] else f"Vein {vein_id}"
            out_names.append(name)

    return jsonify({
        'status': 'success',
        'contour_count': len(polygons),
        'contours': polygons,
        'pink_percentages': pink_percentages,
        'seed_names': out_names
    })

# SERVER LAUNCH
ngrok.kill()
ngrok_config = os.path.expanduser("~/.config/ngrok/ngrok.yml")
if os.path.exists(ngrok_config):
    os.remove(ngrok_config)

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
threading.Thread(target=lambda: app.run(host='0.0.0.0', port=5000), daemon=True).start()

tunnel = ngrok.connect(5000, url=NGROK_DOMAIN)
print(f"PERMANENT ENDPOINT ACTIVE: {tunnel.public_url}/analyze")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


PERMANENT ENDPOINT ACTIVE: https://upstage-justice-oblong.ngrok-free.dev/analyze
